# QuantumCrop AI — Hugging Face → MobileNetV2 Training

**Goal:** Download the official PlantVillage dataset directly from Hugging Face, inspect it, create a leakage-safe validation split from the official training data, train a real MobileNetV2 baseline on Colab GPU, and evaluate only once on the untouched official test set.

Dataset: `mohanty/PlantVillage` (Hugging Face). The notebook never fabricates metrics.


## 0. Colab runtime

Use **Runtime → Change runtime type → GPU**. A T4/L4/A100 is preferred. The notebook uses PyTorch and torchvision.


In [1]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


Sun Aug 23 14:55:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip -q install -U datasets huggingface_hub scikit-learn pandas matplotlib seaborn tqdm


## 1. Download PlantVillage from Hugging Face

Run this cell. No dataset ZIP is required.


In [1]:
!pip install -q "pyarrow>=14.0.1" "datasets>=2.16.0"

from datasets import load_dataset

# Load a verified dataset that contains the actual images and labels
dataset = load_dataset("BrandonFors/Plant-Diseases-PlantVillage-Dataset")

print(dataset)
print('Columns:', dataset['train'].column_names)
print(dataset['train'].features)

README.md:   0%|          | 0.00/6.64k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  321MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  362MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  170MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43456 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10849 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 43456
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10849
    })
})
Columns: ['image', 'label']
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybe

## 2. Inspect labels and class balance


In [2]:
from collections import Counter

# Accessing labels from the valid dataset structure
label_names = dataset['train'].features['label'].names
train_counts = Counter(dataset['train']['label'])

# This repository uses 'test' for its secondary split
test_counts = Counter(dataset['test']['label'])

print('Number of classes:', len(label_names))
for idx, name in enumerate(label_names):
    print(f'{idx:02d} | {name:45s} train={train_counts[idx]:5d} test={test_counts[idx]:5d}')

Number of classes: 38
00 | Apple___Apple_scab                            train=  504 test=  126
01 | Apple___Black_rot                             train=  497 test=  124
02 | Apple___Cedar_apple_rust                      train=  220 test=   55
03 | Apple___healthy                               train= 1316 test=  329
04 | Blueberry___healthy                           train= 1202 test=  300
05 | Cherry_(including_sour)___Powdery_mildew      train=  842 test=  210
06 | Cherry_(including_sour)___healthy             train=  684 test=  170
07 | Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot train=  411 test=  102
08 | Corn_(maize)___Common_rust_                   train=  954 test=  238
09 | Corn_(maize)___Northern_Leaf_Blight           train=  788 test=  197
10 | Corn_(maize)___healthy                        train=  930 test=  232
11 | Grape___Black_rot                             train=  944 test=  236
12 | Grape___Esca_(Black_Measles)                  train= 1107 test=  276
13 | Grape_

## 3. Verify leaf-level grouping

`leaf_id` is important: we use it to make sure a physical leaf is not represented in both train and validation. The official test split remains untouched.


In [5]:
from sklearn.model_selection import train_test_split
import numpy as np

train_indices = np.arange(len(dataset['train']))
train_labels = np.array(dataset['train']['label'])

train_idx, val_idx = train_test_split(
    train_indices,
    test_size=0.15,
    random_state=42,
    stratify=train_labels,
)

print('Train:', len(train_idx))
print('Validation:', len(val_idx))
print('Official test (split="test"):', len(dataset['test']))

Train: 36937
Validation: 6519
Official test (split="test"): 10849


## 4. Build a stratified validation split from TRAIN only

Official test data is never used for tuning.


In [6]:
# ... [Keep your transforms and HFPlantVillage class exactly as they are] ...

train_ds = HFPlantVillage(dataset['train'], train_idx, train_tfms)
val_ds = HFPlantVillage(dataset['train'], val_idx, eval_tfms)

# Update this specific line to point to the correct 'test' split
test_ds = HFPlantVillage(dataset['test'], None, eval_tfms)

BATCH_SIZE = 64
workers = min(4, os.cpu_count() or 2)
# ... [Keep the rest of your DataLoader setup code exactly the same] ...

## 5. Create PyTorch datasets

Images are resized to 224×224. Training gets standard augmentation; validation/test use deterministic preprocessing.


In [7]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class HFPlantVillage(Dataset):
    def __init__(self, hf_split, indices=None, transform=None):
        self.split = hf_split
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = self.split[int(self.indices[i])]
        image = row['image'].convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(row['label'])

# Dataset instances
train_ds = HFPlantVillage(dataset['train'], train_idx, train_tfms)
val_ds = HFPlantVillage(dataset['train'], val_idx, eval_tfms)
test_ds = HFPlantVillage(dataset['test'], None, eval_tfms)

BATCH_SIZE = 64
workers = 0

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=workers, pin_memory=True)

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}, Test size: {len(test_ds)}")

Train size: 36937, Val size: 6519, Test size: 10849


## 6. Train MobileNetV2 — stage 1 transfer learning

Backbone is frozen first. Only the new classifier is trained. This is the fast baseline.


In [10]:
import os, time, copy, json
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(label_names)

model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
for p in model.features.parameters():
    p.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

print('Device:', device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 120MB/s]


Device: cuda
Trainable parameters: 48678


## 7. Training loop

Best checkpoint is selected using validation macro-F1. The official test set is not touched here.


In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

def run_epoch(loader, training=False):
    model.train(training)
    total_loss, y_true, y_pred = 0.0, [], []
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        if training:
            loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        y_true.extend(y.detach().cpu().numpy())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy())
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'loss': total_loss/len(loader.dataset), 'accuracy': acc, 'precision_macro': p, 'recall_macro': r, 'f1_macro': f1}

OUT = 'research/models'
os.makedirs(OUT, exist_ok=True)
best_f1 = -1
history = []
EPOCHS = 8

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    row = {'epoch': epoch, 'train': tr, 'val': va, 'seconds': time.time()-t0}
    history.append(row)
    print(f"Epoch {epoch}/{EPOCHS} | train f1={tr['f1_macro']:.4f} | val f1={va['f1_macro']:.4f} | val acc={va['accuracy']:.4f}")
    if va['f1_macro'] > best_f1:
        best_f1 = va['f1_macro']
        torch.save({'model_state_dict': model.state_dict(), 'label_names': label_names}, f'{OUT}/mobilenetv2_best.pt')
        print('  saved best checkpoint')

json.dump(history, open(f'{OUT}/cnn_history.json','w'), indent=2)


  0%|          | 0/578 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

Epoch 1/8 | train f1=0.8357 | val f1=0.9330 | val acc=0.9468
  saved best checkpoint


  0%|          | 0/578 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

...

  0%|          | 0/102 [00:00<?, ?it/s]

Epoch 7/8 | train f1=0.9338 | val f1=0.9492 | val acc=0.9604


  0%|          | 0/578 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

Epoch 8/8 | train f1=0.9355 | val f1=0.9477 | val acc=0.9592


## 8. Optional stage 2 — fine-tune the last MobileNetV2 layers

Only do this after the baseline completes successfully. This improves adaptation while keeping compute manageable.

In [12]:
# Unfreeze the last ~30 feature layers for controlled fine-tuning.
for p in model.features[-30:].parameters(): p.requires_grad = True
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)

FT_EPOCHS = 4
for epoch in range(1, FT_EPOCHS+1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    print(f"Fine-tune {epoch}/{FT_EPOCHS} | train f1={tr['f1_macro']:.4f} | val f1={va['f1_macro']:.4f} | val acc={va['accuracy']:.4f}")
    if va['f1_macro'] > best_f1:
        best_f1 = va['f1_macro']
        torch.save({'model_state_dict': model.state_dict(), 'label_names': label_names}, f'{OUT}/mobilenetv2_best.pt')


  0%|          | 0/578 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

Fine-tune 1/4 | train f1=0.9684 | val f1=0.9845 | val acc=0.9902


  0%|          | 0/578 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

...

  0%|          | 0/102 [00:00<?, ?it/s]

Fine-tune 3/4 | train f1=0.9926 | val f1=0.9917 | val acc=0.9956


  0%|          | 0/578 [00:00<?, ?it/s]

  0%|          | 0/102 [00:00<?, ?it/s]

Fine-tune 4/4 | train f1=0.9952 | val f1=0.9932 | val acc=0.9966


## 9. Final evaluation — untouched official test set

This is the first and only point where the official test set is used for model reporting.

In [13]:
from sklearn.metrics import classification_report, confusion_matrix

ckpt = torch.load(f'{OUT}/mobilenetv2_best.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in tqdm(test_loader):
        logits = model(x.to(device, non_blocking=True))
        y_true.extend(y.numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())

report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

test_metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'precision_macro': report['macro avg']['precision'],
    'recall_macro': report['macro avg']['recall'],
    'f1_macro': report['macro avg']['f1-score'],
    'num_test_samples': len(y_true),
    'num_classes': num_classes,
    'checkpoint': f'{OUT}/mobilenetv2_best.pt'
}
json.dump(test_metrics, open(f'{OUT}/cnn_test_metrics.json','w'), indent=2)
print(json.dumps(test_metrics, indent=2))


  0%|          | 0/170 [00:00<?, ?it/s]

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.99      0.99      0.99       126
                                 Apple___Black_rot       1.00      1.00      1.00       124
                          Apple___Cedar_apple_rust       1.00      1.00      1.00        55
                                   Apple___healthy       1.00      0.99      1.00       329
                               Blueberry___healthy       1.00      1.00      1.00       300
          Cherry_(including_sour)___Powdery_mildew       1.00      1.00      1.00       210
                 Cherry_(including_sour)___healthy       1.00      1.00      1.00       170
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.97      0.75      0.85       102
                       Corn_(maize)___Common_rust_       1.00      1.00      1.00       238
               Corn_(maize)___Northern_Leaf_Blight       0.88      0.99      0.

## 10. Save split metadata and download the model


In [14]:
split_meta = {
    'dataset': 'BrandonFors/Plant-Diseases-PlantVillage-Dataset',
    'random_state': 42,
    'validation_fraction_from_official_train': 0.15,
    'train_samples': len(train_idx),
    'validation_samples': len(val_idx),
    'official_test_samples': len(test_ds),
    'classes': label_names,
}
json.dump(split_meta, open(f'{OUT}/split_manifest.json', 'w'), indent=2)

from google.colab import files
for f in ['mobilenetv2_best.pt', 'cnn_history.json', 'cnn_test_metrics.json', 'split_manifest.json']:
    files.download(f'{OUT}/{f}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next phase

After the CNN checkpoint is verified, we will **not retrain the CNN locally**. We will use this checkpoint to extract 1280-D MobileNetV2 features, fit PCA on training data only, reduce to 4 quantum features, and train the VQC on the cloud.

Do not claim quantum advantage until CNN, VQC, and hybrid models have been evaluated on the same untouched test set.